# Merge & Aggregate Dataset

## 1. Konfigurasi

In [1]:
import csv
import datetime
from pathlib import Path


def find_base_dir(start=None) -> Path:
    """Cari root repo — folder pertama ke atas yang berisi `dataset/csv/`."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "csv").is_dir():
            return candidate
    raise RuntimeError(f"Root repo tidak ditemukan dari {start}")


BASE_DIR = find_base_dir()

DATE_FORMAT = "%d %b %Y"
EXPECTED_FIELD_COUNT = 7  # Tanggal .. Kuantitas
GROUP_FIELD_COUNT = 6     # 6 kolom pertama = kunci unik; kolom ke-7 (Kuantitas) dijumlahkan

FIELDNAMES = [
    "Tanggal", "Kategori Barang", "Kode Barang", "Nama Barang",
    "Nama Cabang", "Satuan", "Kuantitas",
]

SOURCE_FILES = [
    BASE_DIR / "dataset/csv/jan-24.csv",
    BASE_DIR / "dataset/csv/feb-24.csv",
    BASE_DIR / "dataset/csv/mar-24.csv",
    BASE_DIR / "dataset/csv/apr-des-24.csv",
    BASE_DIR / "dataset/csv/jan-des-25.csv",
]

OUTPUT_FILE = BASE_DIR / "dataset/csv/dataset.csv"

print(f"BASE_DIR    = {BASE_DIR}")
print(f"OUTPUT_FILE = {OUTPUT_FILE}")
for path in SOURCE_FILES:
    print(f"  {'ok ' if path.exists() else 'HILANG'} {path.relative_to(BASE_DIR)}")

BASE_DIR    = /Users/ramapdp/Project/Personal/forecast-scm
OUTPUT_FILE = /Users/ramapdp/Project/Personal/forecast-scm/dataset/csv/dataset.csv
  ok  dataset/csv/jan-24.csv
  ok  dataset/csv/feb-24.csv
  ok  dataset/csv/mar-24.csv
  ok  dataset/csv/apr-des-24.csv
  ok  dataset/csv/jan-des-25.csv


## 2. Parsing & validasi baris

In [2]:
def parse_tanggal(value: str) -> datetime.date:
    """Ubah string tanggal format '01 Jan 2024' menjadi objek datetime.date."""
    return datetime.datetime.strptime(value.strip(), DATE_FORMAT).date()


def normalize_row(row: list[str]) -> list[str]:
    """Pastikan baris punya tepat EXPECTED_FIELD_COUNT kolom.

    Kolom ekstra yang tidak kosong dianggap corrupt dan akan raise ValueError,
    karena itu berarti delimiter salah atau ada field yang tidak diharapkan.
    """
    if len(row) < EXPECTED_FIELD_COUNT:
        raise ValueError(f"Row has fewer than {EXPECTED_FIELD_COUNT} fields: {row!r}")
    kept, extra = row[:EXPECTED_FIELD_COUNT], row[EXPECTED_FIELD_COUNT:]
    if any(field.strip() for field in extra):
        raise ValueError(f"Unexpected non-empty trailing field(s) in row: {row!r}")
    return kept

In [3]:
assert parse_tanggal("01 Jan 2024") == datetime.date(2024, 1, 1)
assert parse_tanggal("31 Dec 2025") == datetime.date(2025, 12, 31)

baris_9_kolom = [
    "01 Jan 2025", "Minuman - FG", "FGS-00014", "Club Mineral 600 ml",
    "KY003 - Kebuli Yaman Serang", "Botol", "4", "", "",
]
assert normalize_row(baris_9_kolom) == baris_9_kolom[:7]

try:
    normalize_row(baris_9_kolom[:7] + ["tak terduga", ""])
except ValueError as e:
    print(f"ValueError seperti yang diharapkan: {e}")
else:
    raise AssertionError("baris dengan kolom ekstra tidak kosong seharusnya raise")

ValueError seperti yang diharapkan: Unexpected non-empty trailing field(s) in row: ['01 Jan 2025', 'Minuman - FG', 'FGS-00014', 'Club Mineral 600 ml', 'KY003 - Kebuli Yaman Serang', 'Botol', '4', 'tak terduga', '']


## 3. Baca & tulis CSV

In [4]:
def read_rows(path) -> list[list[str]]:
    """Baca semua baris data dari CSV (delimiter titik koma), skip header.

    Baris kosong dibuang. Setiap baris divalidasi oleh normalize_row sebelum
    dikembalikan, sehingga caller bisa langsung menggunakan indeks kolom.
    """
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f, delimiter=";")
        next(reader)  # skip header
        return [
            normalize_row(row)
            for row in reader
            if any(field.strip() for field in row)
        ]


def write_rows(rows, path) -> None:
    """Tulis baris data ke CSV dengan header standar FIELDNAMES."""
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(FIELDNAMES)
        writer.writerows(rows)

## 4. Merge & pengurutan

In [5]:
def merge_and_sort(paths) -> list[list[str]]:
    """Gabungkan semua file CSV sumber lalu urutkan kronologis berdasarkan Tanggal.

    Pengurutan dilakukan pada kolom pertama (Tanggal) setelah di-parse ke
    datetime.date, bukan sebagai string, sehingga '01 Feb 2024' < '01 Mar 2024'
    secara benar meski secara alfabet '01 Feb...' > '01 Jan...'.
    """
    all_rows: list[list[str]] = []
    for path in paths:
        all_rows.extend(read_rows(path))
    all_rows.sort(key=lambda row: parse_tanggal(row[0]))
    return all_rows

## 5. Jalankan merge

In [6]:
rows = merge_and_sort(SOURCE_FILES)
write_rows(rows, OUTPUT_FILE)
print(f"Wrote {len(rows)} rows to {OUTPUT_FILE}")

rows[:3]

Wrote 1548269 rows to /Users/ramapdp/Project/Personal/forecast-scm/dataset/csv/dataset.csv


[['01 Jan 2024',
  'Barang Semi FG (WIP-2)',
  'FGS-00001',
  'Ayam Kebuli (0.9)',
  'KY001 - Kebuli Yaman Kutabumi (Pusat)',
  'Potong',
  '220,0'],
 ['01 Jan 2024',
  'Barang Semi FG (WIP-2)',
  'FGS-00001',
  'Ayam Kebuli (0.9)',
  'KY001 - Kebuli Yaman Kutabumi (Pusat)',
  'Potong',
  '1,0'],
 ['01 Jan 2024',
  'Barang Semi FG (WIP-2)',
  'FGS-00001',
  'Ayam Kebuli (0.9)',
  'KY001 - Kebuli Yaman Kutabumi (Pusat)',
  'Potong',
  '2,0']]

## 6. Agregasi baris duplikat

In [7]:
def parse_kuantitas(value: str) -> float:
    """Ubah string kuantitas (koma sebagai desimal) menjadi float.

    Dataset sumber menggunakan koma sebagai pemisah desimal ('1,5' bukan '1.5'),
    sehingga perlu diganti sebelum dikonversi ke float.
    """
    return float(value.replace(",", "."))


def aggregate_rows(rows: list[list[str]]) -> list[list[str]]:
    """Jumlahkan kuantitas untuk baris-baris dengan kunci (item, cabang, tanggal) yang sama.

    Baris duplikat muncul ketika satu item terjual beberapa kali di cabang yang
    sama pada tanggal yang sama dan dicatat sebagai baris terpisah di file sumber.
    Fungsi ini melipatnya menjadi satu baris per kunci unik.

    GROUP_FIELD_COUNT = 6 kolom pertama membentuk kunci unik; kolom ke-7
    adalah Kuantitas yang dijumlahkan.
    """
    totals: dict[tuple[str, ...], float] = {}
    for row in rows:
        key = tuple(row[:GROUP_FIELD_COUNT])
        totals[key] = totals.get(key, 0.0) + parse_kuantitas(row[GROUP_FIELD_COUNT])
    return [list(key) + [str(round(total, 1))] for key, total in totals.items()]

## 7. Jalankan agregasi

In [8]:
aggregated = aggregate_rows(rows)
write_rows(aggregated, OUTPUT_FILE)
print(f"Aggregated {len(rows)} rows into {len(aggregated)} rows, wrote to {OUTPUT_FILE}")

Aggregated 1548269 rows into 693563 rows, wrote to /Users/ramapdp/Project/Personal/forecast-scm/dataset/csv/dataset.csv


## 8. QA hasil akhir

In [9]:
hasil = read_rows(OUTPUT_FILE)

assert len(hasil) == len(aggregated), "jumlah baris di file ≠ hasil agregasi"

kunci = [tuple(row[:GROUP_FIELD_COUNT]) for row in hasil]
assert len(set(kunci)) == len(kunci), "masih ada kunci duplikat setelah agregasi"

tanggal = [parse_tanggal(row[0]) for row in hasil]
assert tanggal == sorted(tanggal), "baris tidak urut kronologis"

print(f"{len(hasil):,} baris  |  {tanggal[0]} … {tanggal[-1]}")
print(f"{len({row[2] for row in hasil}):,} SKU  |  {len({row[4] for row in hasil}):,} cabang")
print(f"Kuantitas total: {sum(parse_kuantitas(row[6]) for row in hasil):,.1f}")

693,563 baris  |  2024-01-01 … 2025-12-31
109 SKU  |  67 cabang
Kuantitas total: 21,077,721.0


## 9. *(Opsional)* Cek sinkron dengan `utils/`

In [10]:
import inspect
import sys

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from utils.merge_split_data import aggregate_dataset as _ref_agg
from utils.merge_split_data import merge_dataset as _ref_merge

PASANGAN = [
    (parse_tanggal, _ref_merge.parse_tanggal),
    (normalize_row, _ref_merge.normalize_row),
    (read_rows, _ref_merge.read_rows),
    (write_rows, _ref_merge.write_rows),
    (merge_and_sort, _ref_merge.merge_and_sort),
    (parse_kuantitas, _ref_agg.parse_kuantitas),
    (aggregate_rows, _ref_agg.aggregate_rows),
]


beda = [
    nb.__name__
    for nb, ref in PASANGAN
    if inspect.getsource(nb) != inspect.getsource(ref)
]

if beda:
    print("BERBEDA dari utils/ — salin ulang atau samakan: " + ", ".join(beda))
else:
    print(f"Sinkron: {len(PASANGAN)} fungsi identik dengan utils/merge_split_data/")

Sinkron: 7 fungsi identik dengan utils/merge_split_data/
